# Module 6 studio — Reproducible pipeline
**PUBHLT 0411 · Python for Public Health Data Analysis**

Name: 

Submit this notebook as a `.ipynb` file.

---

Work each exercise in the cell provided. Replace every blank (`____`) with working code, then run the cell to check it does what the task asks.

## Before you start — load the data

This lab uses:

- `pa_counties.csv`
- `pa_firearm_county_year.csv`

Run the setup cell below **once**, at the start of the session. It downloads the files from the course site into a `data/` folder next to this notebook.

Nothing to upload, and no Google Drive needed. If the Colab session restarts, run it again.

In [ ]:
# Setup — run this cell once, before anything else.
#
# Downloads the lab's CSVs from the course site into a local `data/` folder.
# Nothing to upload and no Google Drive needed. Re-run it if the session restarts.
import os
import urllib.request

COURSE_DATA = "https://soumikp.github.io/pubhlt0411-python/data/"

FILES = [
    "pa_counties.csv",
    "pa_firearm_county_year.csv"
]

os.makedirs("data", exist_ok=True)

for name in FILES:
    target = os.path.join("data", name)
    if not os.path.exists(target):
        urllib.request.urlretrieve(COURSE_DATA + name, target)

print("Data ready:", ", ".join(FILES))

---

## Exercise 1 — Import and inspect

**Worked example.** A pipeline opens by loading its inputs and confirming their shape. An
analysis that never checks what arrived is an analysis that trusts a file it has not read.

In [ ]:
import pandas as pd

firearm = pd.read_csv("data/pa_firearm_county_year.csv")
counties = pd.read_csv("data/pa_counties.csv")

print(firearm.shape, counties.shape)
print(firearm.columns.tolist())
print(counties.columns.tolist())

In [ ]:
print(firearm["firearm_deaths"].isna().sum(), "of", len(firearm), "rows suppressed")
print(firearm[firearm["year"] == 2023]["firearm_deaths"].isna().sum(), "of 67 in 2023")

In [ ]:
demo_2023 = firearm[firearm["year"] == 2023]
demo_reporting = demo_2023.dropna(subset=["firearm_deaths"])

demo_joined = demo_reporting.merge(
    counties[["county", "metro", "region", "population_2023"]],
    on="county", how="left"
)

demo_summary = demo_joined.groupby("metro").agg(
    counties=("county", "size"),
    deaths=("firearm_deaths", "sum"),
    population=("population_2023", "sum"),
)
demo_summary["rate_per_100k"] = demo_summary["deaths"] / demo_summary["population"] * 100000
print(demo_summary.round(1))

In [ ]:
demo_naive = demo_2023.merge(counties[["county", "metro", "population_2023"]],
                             on="county", how="left")
demo_wrong = demo_naive.groupby("metro").agg(
    deaths=("firearm_deaths", "sum"),
    population=("population_2023", "sum"),
)
demo_wrong["rate_per_100k"] = demo_wrong["deaths"] / demo_wrong["population"] * 100000
print(demo_wrong.round(1))

In [ ]:
demo_joined["suicides"] = demo_joined["firearm_suicide"].fillna(0)

demo_share = demo_joined.groupby("metro").agg(
    deaths=("firearm_deaths", "sum"),
    suicides=("suicides", "sum"),
)
demo_share["suicide_share_pct"] = demo_share["suicides"] / demo_share["deaths"] * 100
print(demo_share.round(1))

In [ ]:
import matplotlib.pyplot as plt

allyears = firearm.dropna(subset=["firearm_deaths"]).merge(
    counties[["county", "metro"]], on="county", how="left"
)
trend = (allyears.groupby(["year", "metro"])
         .apply(lambda g: g["firearm_deaths"].sum() / g["population"].sum() * 100000)
         .unstack())

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(trend.index, trend["Nonmetro"], marker="^", linewidth=2.5,
        color="#8a6400", label="Nonmetro")
ax.plot(trend.index, trend["Metro"], marker="o", linewidth=2.5,
        color="#003594", label="Metro")
ax.set_xlabel("Year")
ax.set_ylabel("Firearm deaths per 100,000")
ax.set_title("Pooled firearm death rate, reporting PA counties, 2019–2024")
ax.set_ylim(0, 22)
ax.legend()
plt.tight_layout()
plt.show()

print(trend.round(1))

In [ ]:
import pandas as pd

firearm = pd.read_csv("data/pa_firearm_county_year.csv")

print(firearm.shape)
print(firearm["year"].min(), "to", firearm["year"].max())
print(firearm["firearm_deaths"].isna().sum(), "suppressed cells")

**Your turn.** Load `data/pa_counties.csv` into `county_reference`. Print its shape, how many
counties fall into each `metro` category, and confirm the file has one row per county.

In [ ]:
import pandas as pd

county_reference = pd.______("data/pa_counties.csv")

print(county_reference.______)
print(county_reference["metro"].______())
print("one row per county:", county_reference["county"].______() == len(county_reference))

---

## Exercise 2 — Filter to one year

**Worked example.** The file holds six years stacked in one column. A question about 2023
requires selecting those rows before anything else happens.

In [ ]:
f2021 = firearm[firearm["year"] == 2021]
print(f2021.shape)
print(f2021["firearm_deaths"].isna().sum(), "suppressed in 2021")

**Your turn.** Build `f2023`, the 2023 rows only. Print its shape, how many `firearm_deaths`
values are missing, and what fraction of counties that leaves reporting.

In [ ]:
f2023 = ______[______["year"] ______ 2023]

n_missing = f2023["firearm_deaths"].______().sum()
n_reporting = len(f2023) - n_missing

print(f2023.shape)
print(n_missing, "suppressed in 2023")
print("reporting:", n_reporting, "of", len(f2023),
      f"({n_reporting / len(f2023):.0%})")

---

## Exercise 3 — Clean, and decide about the blanks

**Worked example.** `.dropna(subset=[...])` removes rows missing a value in the named column
and leaves every other column alone. Naming the column matters: dropping on the whole frame
would also discard counties whose *homicide* count is suppressed but whose total is known.

In [ ]:
careless = f2023.dropna()
print("dropna() on everything:", len(careless), "counties left")

**Your turn.** Build `reporting`: the 2023 counties that report a `firearm_deaths` total,
dropping on **that column only**. Then show what the careless version costs, by naming the
counties it would discard even though their death total is known.

In [ ]:
reporting = f2023.______(______=["firearm_deaths"])

lost = set(reporting["county"]) - set(f2023.______()["county"])

print(len(reporting), "counties report a 2023 total")
print(reporting["firearm_deaths"].______(), "deaths among them")
print(len(lost), "counties a bare dropna() would have thrown away:")
print(sorted(lost))

---

## Exercise 4 — Join

**Worked example.** `.merge()` attaches columns from a second table by matching on a shared
key. `how="left"` keeps every row of the left table whether or not it finds a match, which
makes an unmatched key visible instead of silent.

In [ ]:
example = reporting.merge(county_reference[["county", "region"]], on="county", how="left")
print(example[["county", "firearm_deaths", "region"]].head(3))
print(example["region"].isna().sum(), "rows failed to match")

**Your turn.** Merge `reporting` with the `county`, `metro`, and `population_2023` columns of
`county_reference`, matching **on county**. Confirm the row count is unchanged and that
nothing failed to match.

In [ ]:
before = len(reporting)

joined = reporting.______(
    county_reference[["county", "metro", "population_2023"]],
    ______="county",
    how="______"
)

print(joined.shape)
print("rows preserved:", len(joined) ______ before)
print(joined["metro"].______().sum(), "rows failed to match")

---

## Exercise 5 — Group and summarise

**Worked example.** `.groupby().agg()` collapses many rows into one row per group, naming
each output column and the operation that fills it. `"size"` counts rows; `"sum"` totals a
column.

In [ ]:
by_region = (reporting
             .merge(county_reference[["county", "region"]], on="county", how="left")
             .groupby("region")
             .agg(counties=("county", "size"),
                  deaths=("firearm_deaths", "sum")))
print(by_region)

**Your turn — no scaffold.** Answer this question from `joined`:

> **Do Pennsylvanians living outside metropolitan counties die by firearm at a higher rate
> than those living in them?**

Produce a table called `summary`, one row per metro group, reporting for each group: **how
many counties it contains, its total firearm deaths, its total population, and its firearm
death rate per 100,000 people.** Print it.

In [ ]:
# Write the whole step yourself.






print(summary.round(1))

---

## Exercise 6 — The error that raises nothing

**Worked example.** The pipeline below is identical to Exercise 5 except for one omission. It
runs without complaint and returns the opposite finding.

In [ ]:
broken = f2023.merge(county_reference[["county", "metro", "population_2023"]],
                     on="county", how="left")
broken_summary = broken.groupby("metro").agg(
    deaths=("firearm_deaths", "sum"),
    population=("population_2023", "sum"),
)
broken_summary["rate_per_100k"] = (broken_summary["deaths"]
                                   / broken_summary["population"] * 100000)
print(broken_summary.round(1))

**Your turn.** Repair it, then measure the damage the broken version did: how many people
were sitting in the nonmetro denominator with no deaths above them, and how far that pushed
the nonmetro rate.

In [ ]:
repaired = f2023.______(______=["firearm_deaths"]).merge(
    county_reference[["county", "metro", "population_2023"]], on="county", how="left"
)

fixed = repaired.groupby("metro").agg(
    deaths=("firearm_deaths", "sum"),
    population=("population_2023", "sum"),
)
fixed["rate_per_100k"] = fixed["deaths"] / fixed["______"] * 100000

phantom = (broken_summary.loc["Nonmetro", "population"]
           ______ fixed.loc["Nonmetro", "population"])
understated_by = (fixed.loc["Nonmetro", "rate_per_100k"]
                  ______ broken_summary.loc["Nonmetro", "rate_per_100k"])

print(fixed.round(1))
print(f"{phantom:,} nonmetro residents were in the denominator with no deaths above them")
print(f"the broken rate understated nonmetro by {understated_by:.1f} per 100,000")

---

## Exercise 7 — Visualise

**Worked example.** A two-bar comparison is the right figure for a two-group finding. The
zero baseline is what keeps the comparison honest, and the value labels remove the need to
read heights against a gridline.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.2))
bars = ax.bar(["Metro", "Nonmetro"],
              [summary.loc["Metro", "rate_per_100k"],
               summary.loc["Nonmetro", "rate_per_100k"]],
              color=["#003594", "#8a6400"], edgecolor="white")
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set_ylim(0, 22)
ax.set_ylabel("Firearm deaths per 100,000")
ax.set_title("Pooled 2023 firearm death rate, 42 reporting PA counties")
plt.tight_layout()
plt.show()

**Your turn — no scaffold.** A reader accepts that the nonmetro rate is higher and asks the
obvious follow-up: **what kind of firearm death is driving it?**

Build a table called `share` giving, for each metro group, its **total firearm deaths, its
total firearm suicides, and suicides as a percentage of firearm deaths.** Treat a suppressed
suicide count as zero. Then draw a figure that lets a reader compare the two percentages, and
print the table.


> **Requirements the figure must meet**  
> A labelled vertical axis · a title · **a baseline at zero** · each bar labelled with its
> value · colours drawn from the course palette (`#003594`, `#8a6400`).

In [ ]:
# Write the whole step yourself: build `share`, then draw the figure.








print(share.round(1))

---

## Exercise 8 — Describe the figure

**Worked example.** A figure that cannot be read aloud is unavailable to part of its audience
and to anyone whose image fails to load. A description states the chart type, the axes, and
the values — not the conclusion.

In [ ]:
description = (
    "Bar chart of the pooled 2023 firearm death rate for the 42 reporting "
    "Pennsylvania counties. The metro bar reaches 14.1 deaths per 100,000 "
    "and the nonmetro bar reaches 18.5. The vertical axis starts at zero."
)
print(description)
print(len(description), "characters")

**Your turn — no scaffold.** Someone using a screen reader will hear your description instead
of seeing the figure you drew in Exercise 7. Write it into a string called
`share_description`, and print it.

They must be able to reconstruct the figure from your words alone: **what kind of chart it
is, what the two groups are, what quantity is being shown, both of its values, and where the
vertical axis begins.** Describe what is on the page, not what it means — the conclusion is
the reader's to draw.

In [ ]:
# Write the description yourself.




print(share_description)

---

## Exercise 9 — Report the finding

**Worked example.** A reported number is assembled from the pipeline's own variables rather
than typed by hand. A number typed by hand stops being a result the moment the data changes.

In [ ]:
n_reporting = len(reporting)
nonmetro_rate = summary.loc["Nonmetro", "rate_per_100k"]

print(f"Among the {n_reporting} reporting counties, the nonmetro rate "
      f"was {nonmetro_rate:.1f} per 100,000.")

**Your turn — no scaffold.** Write the paragraph a reader would see, into a string called
`finding`, and print it.

It must state, **in prose**: how many counties reported a count and out of how many; both
pooled rates, identified by group; how many deaths the nonmetropolitan estimate rests on and
across how many counties; and that counties reporting no total were excluded from both the
numerator and the denominator.


> **Every number must be computed, not typed**  
> Build the sentence with an f-string reading from `summary` and `reporting`. **A number typed
> by hand stops being a result the moment the data changes.**

In [ ]:
# Write the whole paragraph yourself, as one f-string.






print(finding)